# Loss functions

> A set of custom loss functions

In [ ]:
#| default_exp losses

In [ ]:
#| hide
from nbdev.showdoc import *

# Loss Functions

In [ ]:
#| export
from bioMONAI.core import store_attr

In [ ]:
#| export
import numpy as np

from monai.losses import SSIMLoss
import torch.nn as nn
from torch import sigmoid

from scipy.optimize import curve_fit

from bioMONAI.metrics import FRCMetric, get_fourier_ring_correlations


In [ ]:
show_doc(SSIMLoss)

---

### SSIMLoss

>      SSIMLoss (spatial_dims:int, data_range:float=1.0,
>                kernel_type:monai.metrics.regression.KernelType|str=gaussian,
>                win_size:int|collections.abc.Sequence[int]=11,
>                kernel_sigma:float|collections.abc.Sequence[float]=1.5,
>                k1:float=0.01, k2:float=0.03,
>                reduction:monai.utils.enums.LossReduction|str=mean)

Compute the loss function based on the Structural Similarity Index Measure (SSIM) Metric.

For more info, visit
    https://vicuesoft.com/glossary/term/ssim-ms-ssim/

SSIM reference paper:
    Wang, Zhou, et al. "Image quality assessment: from error visibility to structural
    similarity." IEEE transactions on image processing 13.4 (2004): 600-612.

### Combined Loss

In [ ]:
#| export

class CombinedLoss:
    "losses combined"
    def __init__(self, spatial_dims=2, alpha=0.33, beta=0.33):
        store_attr()
        self.SSIM_loss = SSIMLoss(spatial_dims=spatial_dims)
        self.MSE_loss =  nn.MSELoss()
        self.MAE_loss =  nn.L1Loss()
        
    def __call__(self, pred, targ):
        return (1 - self.alpha - self.beta) * self.SSIM_loss(pred, targ) + self.alpha * self.MSE_loss(pred, targ) + self.beta * self.MAE_loss(pred, targ)
        

### Dice Loss

In [ ]:
#| export

class DiceLoss(nn.Module):

    """
    DiceLoss computes the Sørensen–Dice coefficient loss, which is often used 
    for evaluating the performance of image segmentation algorithms.

    The Dice coefficient is a measure of overlap between two samples. It ranges 
    from 0 (no overlap) to 1 (perfect overlap). The Dice loss is computed as 
    1 - Dice coefficient, so it ranges from 1 (no overlap) to 0 (perfect overlap).

    Attributes:
        smooth (float): A smoothing factor to avoid division by zero and ensure numerical stability.

    Methods:
        forward(inputs, targets):
            Computes the Dice loss between the predicted probabilities (inputs) 
            and the ground truth (targets).
    """

    def __init__(self, smooth=1):

        """
        Initializes the DiceLoss instance with a smoothing factor.

        Args:
            smooth (float): A smoothing factor to avoid division by zero and ensure numerical stability.
                            Default is 1.
        """
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        
        # Make sure the inputs are probabilities
        inputs = sigmoid(inputs)

        # Flatten tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        # Calculate the intersection
        intersection = (inputs * targets).sum()

        # Compute Dice Coefficient
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

        # Copmute dice loss
        loss = 1 - dice

        return loss
        

In [ ]:
# inputs and targets must be equally dimensional tensors
from torch import randn, randint

inputs = randn((1, 1, 256, 256))  # Input
targets = randint(0, 2, (1, 1, 256, 256)).float()  # Ground Truth

# Initialize
dice_loss = DiceLoss()

# Compute loss
loss = dice_loss(inputs, targets)
print('Dice Loss:', loss.item())



Dice Loss: 0.4994280934333801


### Fourier Ring Correlation

In [ ]:
#| export

def FRCLoss(image1, image2):

    """
    Compute the Fourier Ring Correlation (FRC) loss between two images.

    #### Args:
        - image1 (torch.Tensor): The first input image.
        - image2 (torch.Tensor): The second input image.

    #### Returns:
        - torch.Tensor: The FRC loss.
    """
    
    return (1 - FRCMetric(image1, image2))
    

In [ ]:
#| export
def seventh_fourier_ring_correlation(image1,image2):


    """
    Calculate the cutoff frequency at when Fourier ring correlation drops to 1/7.

    #### Args:
        - image1 (torch.Tensor): The first input image.
        - image2 (torch.Tensor): The second input image.

    #### Returns:
        - float: The cutoff frequency.
    """

    # Get y and x coordinates
    y, x = get_fourier_ring_correlations(image1, image2)

    # x -> frequency   y -> correlation
    x = x.numpy()
    y = y.numpy()


    # Exponential function to fit
    def exponential_func(x, a, b, c):
        return a * np.exp(-b * x) + c

    # Make fit
    params, _ = curve_fit(exponential_func, x, y, p0=[1, 1, 1])

    # Get Cutoff requency at 1/7
    cutoff_frequency = (exponential_func((1/7), *params))

    return cutoff_frequency

In [ ]:
show_doc(seventh_fourier_ring_correlation)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L241){target="_blank" style="float:right; font-size:smaller"}

### seventh_fourier_ring_correlation

>      seventh_fourier_ring_correlation (image1, image2)

Calculate the cutoff frequency at when Fourier ring correlation drops to 1/7.

#### Args:
    - image1 (torch.Tensor): The first input image.
    - image2 (torch.Tensor): The second input image.

#### Returns:
    - float: The cutoff frequency.

---

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()